In [2]:
import os
import re
import random
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Pour la reproductibilité des tirages aléatoires
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
RAW_DEBATES_DIR = PROJECT_ROOT / "data" / "kialo" / "raw" / "debates"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dossier des débats bruts : {RAW_DEBATES_DIR}")
print(f"Dossier de sortie : {PROCESSED_DIR}")

Dossier des débats bruts : /mnt/c/Users/mel77/OneDrive/Documents/COURS/M2/DYCO/dyco/src/data/kialo/raw/debates
Dossier de sortie : /mnt/c/Users/mel77/OneDrive/Documents/COURS/M2/DYCO/dyco/src/data/processed


In [3]:
def parse_kialo_file(file_path):
    """
    Lit un fichier texte d'export Kialo et extrait l'arborescence des arguments,
    en gérant les textes qui s'étalent sur plusieurs lignes.
    """
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.readlines()

    nodes = {}  
    title = ""
    current_node_id = None # Pour garder en mémoire l'argument en cours de lecture

    # Expression régulière pour détecter le préfixe Kialo (ex: "1.1.2. Pro: ", "1. Con: ")
    node_pattern = re.compile(r"^([\d\.]+)\s+(Pro|Con):\s*(.*)$", re.IGNORECASE)

    for line in lines:
        clean_line = line.strip()
        if not clean_line:
            continue

        match = node_pattern.match(clean_line)
        if match:
            # Nouveau nœud détecté
            node_id_str, stance, text = match.groups()
            node_id_str = node_id_str.rstrip(".")
            
            parts = node_id_str.split(".")
            if len(parts) == 1:
                parent_id = "root"
            else:
                parent_id = ".".join(parts[:-1])

            nodes[node_id_str] = {
                "id": node_id_str,
                "parent_id": parent_id,
                "stance": stance.capitalize(),
                "text": text.strip()
            }
            current_node_id = node_id_str # On met à jour le nœud actif
            
        elif current_node_id:
            # Si la ligne ne commence pas par un tag Pro/Con, c'est la suite du texte du nœud en cours
            nodes[current_node_id]["text"] += " " + clean_line
            
        elif not title and not clean_line.startswith("http"):
            # Thèse principale du débat
            title = clean_line

    return title, nodes

In [4]:
def extract_pairs_from_debate(debate_id, title, nodes, negative_ratio=1.0):
    """
    Génère les paires d'arguments positives et négatives pour un débat.
    """
    pairs = []
    
    # Paires Positives (Parent - Enfant direct)
    positive_pairs = []
    node_keys = list(nodes.keys())
    
    for node_id, node in nodes.items():
        parent_id = node["parent_id"]
        
        # Si le parent existe dans nos nœuds
        if parent_id in nodes:
            parent_text = nodes[parent_id]["text"]
            child_text = node["text"]
            relation = "Support" if node["stance"] == "Pro" else "Attack"
            
            positive_pairs.append({
                "debate_id": debate_id,
                "arg1": parent_text,      # Argument cible / Cible
                "arg2": child_text,       # Argument source / Attaquant ou Supporter
                "label": relation,
                "arg1_id": parent_id,
                "arg2_id": node_id
            })
            
    pairs.extend(positive_pairs)
    num_positives = len(positive_pairs)

    # Paires Négatives Intra-débat (No-Relation)
    intra_negatives = []
    if len(node_keys) > 2:
        max_attempts = num_positives * 3
        attempts = 0
        
        while len(intra_negatives) < int(num_positives * negative_ratio) and attempts < max_attempts:
            attempts += 1
            id1, id2 = random.sample(node_keys, 2)
            
            # Pour s'assurer qu'il n'y ai pas de lien direct parent-enfant
            if nodes[id1]["parent_id"] != id2 and nodes[id2]["parent_id"] != id1:
                intra_negatives.append({
                    "debate_id": debate_id,
                    "arg1": nodes[id1]["text"],
                    "arg2": nodes[id2]["text"],
                    "label": "No-Relation",
                    "arg1_id": id1,
                    "arg2_id": id2
                })
                
    pairs.extend(intra_negatives)
    return pairs

In [5]:
all_pairs = []
files = list(RAW_DEBATES_DIR.glob("*.txt"))

print(f"Analyse de {len(files)} fichiers de débats...")

for file_path in tqdm(files, desc="Extraction des paires"):
    debate_id = file_path.stem
    title, nodes = parse_kialo_file(file_path)
    
    if nodes:
        debate_pairs = extract_pairs_from_debate(debate_id, title, nodes, negative_ratio=1.0)
        all_pairs.extend(debate_pairs)

df_pairs = pd.DataFrame(all_pairs)
print(f"\nTotal paires extraites : {len(df_pairs)}")

Analyse de 250 fichiers de débats...


Extraction des paires:   0%|          | 0/250 [00:00<?, ?it/s]


Total paires extraites : 193122


In [6]:
# Supprimer les doublons et les textes trop courts (< 10 caractères)
df_pairs = df_pairs.drop_duplicates(subset=["arg1", "arg2"]).copy()
df_pairs = df_pairs[(df_pairs["arg1"].str.len() > 10) & (df_pairs["arg2"].str.len() > 10)].copy()

# Statistiques sur la répartition des classes
print("--- Répartition des classes ---")
print(df_pairs["label"].value_counts())
print("\n--- Pourcentages ---")
print(df_pairs["label"].value_counts(normalize=True) * 100)

--- Répartition des classes ---
label
No-Relation    96445
Attack         48689
Support        47810
Name: count, dtype: int64

--- Pourcentages ---
label
No-Relation    49.986006
Attack         25.234783
Support        24.779211
Name: proportion, dtype: float64


In [7]:
output_csv_path = PROCESSED_DIR / "kialo_pairs.csv"
df_pairs.to_csv(output_csv_path, index=False, encoding="utf-8")

print(f"Dataset sauvegardé avec succès dans : {output_csv_path}")
print(f"Taille totale du jeu de données : {len(df_pairs)} paires d'arguments.")

Dataset sauvegardé avec succès dans : /mnt/c/Users/mel77/OneDrive/Documents/COURS/M2/DYCO/dyco/src/data/processed/kialo_pairs.csv
Taille totale du jeu de données : 192944 paires d'arguments.


In [8]:
# Aperçu des premières paires
df_pairs.head(10)

,debate_id,arg1,arg2,label,arg1_id,arg2_id
0,according-to-the-bible-salvation-is-by-faith-a...,Baptism is also required for salvation.,"Mark 16:16 [1] says that ""Whoever believes an...",Support,1.1,1.1.1
1,according-to-the-bible-salvation-is-by-faith-a...,"Mark 16:16 [1] says that ""Whoever believes an...",The portion of Mark where this verse appears ...,Attack,1.1.1,1.1.1.1
2,according-to-the-bible-salvation-is-by-faith-a...,"Mark 16:16 [1] says that ""Whoever believes an...",This passage may imply that all who believe a...,Attack,1.1.1,1.1.1.2
3,according-to-the-bible-salvation-is-by-faith-a...,Baptism is also required for salvation.,"Galatians 3:27 [3] says that ""all of you who ...",Support,1.1,1.1.2
4,according-to-the-bible-salvation-is-by-faith-a...,"Galatians 3:27 [3] says that ""all of you who ...",This passage is not necessarily referring to ...,Attack,1.1.2,1.1.2.1
5,according-to-the-bible-salvation-is-by-faith-a...,This passage is not necessarily referring to ...,1 Corinthians 12:13 [4] discusses a nonlitera...,Support,1.1.2.1,1.1.2.1.1
6,according-to-the-bible-salvation-is-by-faith-a...,1 Corinthians 12:13 [4] discusses a nonlitera...,The Holy Spirit is closely associated [5] wit...,Attack,1.1.2.1.1,1.1.2.1.1.1
7,according-to-the-bible-salvation-is-by-faith-a...,1 Corinthians 12:13 [4] discusses a nonlitera...,If baptism is what adds [7] one to the church...,Attack,1.1.2.1.1,1.1.2.1.1.2
8,according-to-the-bible-salvation-is-by-faith-a...,This passage is not necessarily referring to ...,Ephesians 4:15 shows that there is only one b...,Attack,1.1.2.1,1.1.2.1.2
9,according-to-the-bible-salvation-is-by-faith-a...,Baptism is also required for salvation.,"1 Peter 3:21 [8] refers to the ""baptism that ...",Support,1.1,1.1.3
